# CT.M1 – Data Acquisition
### Notebook 04. Molecular Representation & Visualization

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/04_molecular_representation.ipynb)


---


### Contents

This notebook serves as the **Visual Analytics and Structural Audit** stage of our drug discovery workflow. Now that we have a curated dataset, we transition from raw data collection to deep scientific exploration, ensuring our chemical library is balanced, clean, and ready for predictive modeling.

This notebook begins with **Step 0: Chemical Library Initialization**, outlining the transition from raw data toward structural interpretation and explaining the importance of connecting numerical descriptors with molecular structure. This is followed by **Step 1: Sanity Check**, where we load the pparg_chembl_ml_dataset.csv and perform a quality control audit on column integrity and missing values.

The analytical core begins with **Step 2: Class Distribution Analysis** and **Step 3: Logarithmic Distribution of Bioactivity ($IC_{50}$)**, where we evaluate the balance between Active and Inactive compounds and verify our activity thresholds. We then move into **Step 4: Comparative Property Analysis** and **Step 5: Multi-dimensional Property Correlation**, examining how Molecular Weight, AlogP, and TPSA distributions reveal preliminary structure–activity relationships and chemical space clusters.

The final phase focuses on **Step 6: Molecular Visualization Engine** and **Step 7: Batch Processing**. Here, we implement a strategy to render 2D structures via SMILES-to-CID conversion and PubChem retrieval. Molecules are displayed in a systematic grid format with pagination to facilitate qualitative structural inspection, allow for the identification of recurring motifs, and generate structural hypotheses for future machine learning modeling.


This notebook is designed to be completed in approximately **120–180 minutes**.


---

### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_04_molecular_representation.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---


## Data Exploration, Quality Control, and Molecular Visualization

The exploration of chemical datasets in drug discovery requires the integration of structural, physicochemical, and biological dimensions of molecular information. While machine learning models operate on numerical representations of molecules, meaningful interpretation depends on reconnecting these abstract descriptors to underlying chemical structures. Consequently, structural visualization and physicochemical profiling serve as the bridge between raw bioactivity data and computational modeling.

In medicinal chemistry, molecular activity is influenced by interrelated factors captured by key descriptors:
* Molecular Weight (MW) and Topology reflect molecular size and complexity.
* Lipophilicity (AlogP) dictates a molecule's ability to partition into biological membranes.
* Topological Polar Surface Area (TPSA) quantifies hydrogen bonding capacity and polarity.

Variations in these parameters reflect differences in membrane permeability, solubility, and binding affinity, defining the "drug-likeness" of the library.

The binary classification of compounds into Active and Inactive categories simplifies biological interpretation while preserving mechanistic relevance. Transforming potency values into a **logarithmic scale** is standard practice; it accounts for the exponential nature of binding equilibria and allows for the clear visualization of distributions across several orders of magnitude.

Finally, two-dimensional structural depictions provide a qualitative perspective on chemical diversity. While 2D representations do not capture full conformational flexibility, they allow for the rapid identification of **recurring scaffolds** and functional group patterns. This visual inspection supports hypothesis generation regarding the structural motifs that drive biological activity, ensuring that computational abstraction remains anchored to fundamental physicochemical principles.

---


### Step 0: Chemical Library Initialization

This block serves as the **Molecular Entry Point**. Before we can visualize or interact with the chemical structures, we must load the curated library into memory. This step is essential for transitioning from abstract data to tangible chemical entities.

This block initializes the graphical environment. In a chemoinformatics workflow, **Matplotlib** is the bridge between raw numerical data and visual scientific insight.

In [ ]:
# Step 0

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_file = "pparg_chembl_ml_dataset.csv"
df = pd.read_csv(csv_file)

print(df.shape)
df.head()


### Step 1: Sanity Check

After loading the chemical library, this step validates the **structural and numerical health** of the data. In drug discovery, "garbage in, garbage out" is a literal risk; this code ensures we are working with high-quality scientific information.

This code teaches students that raw data is rarely perfect. Identifying missing values is the first step toward data imputation or removal, which are core skills in bioinformatics.

By looking at the `min` and `max` of `pchembl_value`, the student can see the **dynamic range** of the bioactivity. A narrow range (e.g., all values between 5.0 and 5.1) would make it impossible to distinguish between active and inactive compounds.

In [ ]:
# Step 1

print("Columns:", df.columns.tolist())
print("\nMissing values per column:\n", df.isna().sum().sort_values(ascending=False))

# Duplicates by molecule
dup = df.duplicated(subset=["molecule_chembl_id"]).sum() if "molecule_chembl_id" in df.columns else None
print("\nDuplicates (molecule_chembl_id):", dup)

# Basic numeric summary
num_cols = [c for c in ["MolecularWeight", "AlogP", "TPSA", "standard_value", "pchembl_value"] if c in df.columns]
df[num_cols].describe()

### Step 2: Class Distribution Analysis

In drug discovery and machine learning, understanding the balance between "Actives" and "Inactives" is critical for determining how we interpret the success of our future models.

Most drug discovery datasets are "imbalanced" (they usually contain many more inactives than actives). This visualization teaches students to recognize this reality. If a dataset has 95% inactives, a model could achieve 95% accuracy by simply guessing "Inactive" every time—a trap students must learn to avoid.

This step creates a visual and numerical profile of our target variable. It allows us to see how the "bioactivity threshold" we set in the previous notebook ($1000\ \text{nM}$) divided our chemical library.

Seeing this distribution helps the researcher decide if they need to adjust their $IC_{50}$ threshold. If there are too few "Actives," the threshold might be too strict for a preliminary study.



In [ ]:
# Step 2 

counts = df["Activity_Class"].value_counts()

plt.figure()
plt.bar(counts.index, counts.values)
plt.title("Class balance (Activity_Class)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

counts

### Step 3: Logarithmic Distribution of Bioactivity (`$IC_{50}$`)

By visualizing the distribution of raw potency values, we move from a simple "Yes/No" classification to understanding the spectrum of drug-target interactions.

This step analyzes the physical range of the $IC_{50}$ values in our library. In pharmacology, potency is often viewed on a logarithmic scale because biological responses span several orders of magnitude (from nanomolar to micromolar).

By looking at the histogram, the researcher can see if their $1,000$ nM threshold sits in a "natural" gap in the data or if it's cutting through a dense cluster of molecules. This helps justify the scientific choice made in the previous notebook.

A "bimodal" distribution (two peaks) might suggest two different types of binding mechanisms or two different assay protocols merged into one table, prompting further investigation.



In [ ]:
# Step 3

ic50 = df["standard_value"].astype(float)

plt.figure()
plt.hist(np.log10(ic50.replace(0, np.nan).dropna()), bins=30)
plt.title("IC50 distribution (log10 nM)")
plt.xlabel("log10(IC50 nM)")
plt.ylabel("Count")
plt.axvline(np.log10(1000), linestyle="--")  # threshold
plt.show()

### Step 4: Comparative Property Analysis

By overlaying the physical properties of "Active" versus "Inactive" molecules, we begin to uncover the Structure-Property Relationships that define a successful drug candidate.

This step uses **Overlaid Histograms** to test scientific hypotheses. We are looking for "Separation": do active molecules tend to be heavier, more lipophilic (AlogP), or more polar (TPSA) than inactive ones?

Students will learn to identify which properties "matter." If the "Active" and "Inactive" histograms perfectly overlap for `MolecularWeight`, then weight is not a good predictor for this target. However, if "Actives" shift toward a specific AlogP range, the student has discovered a Pharmacological Trend.

The **visualization** reinforces the concept of the *Lipinski Rule of 5*. It shows whether the active compounds fall within the traditional "Rule of 5" boundaries or if the PPAR$\gamma$ target requires molecules with atypical properties.

This is the bridge to Machine Learning. If a student sees a clear separation in these plots, they can confidently predict that a model (like a Decision Tree) will perform well using these specific features.

In [ ]:
# Step 4

features = [c for c in ["MolecularWeight", "AlogP", "TPSA"] if c in df.columns]

for feat in features:
    plt.figure()
    for cls in ["Active", "Inactive"]:
        x = df.loc[df["Activity_Class"] == cls, feat].astype(float).dropna()
        plt.hist(x, bins=30, alpha=0.5, label=cls)
    plt.title(f"{feat} distribution by class")
    plt.xlabel(feat)
    plt.ylabel("Count")
    plt.legend()
    plt.show()

### Step 5: Multi-dimensional Property Correlation

By moving from one-dimensional histograms to a two-dimensional **Scatter Plot**, we can visualize how two critical parameters interact to define the "Bioactive Space."

This step analyzes the relationship between **Lipophilicity (AlogP)** and **Polarity (TPSA)**. In drug discovery, these two properties are often inversely correlated and are the primary drivers of a molecule's ability to cross biological membranes.

Students will observe the "trade-off" between solubility (linked to TPSA) and permeability (linked to AlogP). This is a fundamental concept in **ADME (Absorption, Distribution, Metabolism, and Excretion)**.

Points that are "deep" within the Active cluster are potential **Lead Compounds**, while active points that are far away from the cluster might represent Unique Scaffolds with a different binding mode.

In [ ]:
# Step 5

xcol, ycol = "AlogP", "TPSA"
if xcol in df.columns and ycol in df.columns:
    plt.figure()
    for cls in ["Active", "Inactive"]:
        sub = df[df["Activity_Class"] == cls]
        plt.scatter(sub[xcol], sub[ycol], alpha=0.7, label=cls)
    plt.title(f"{ycol} vs {xcol}")
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.legend()
    plt.show()
else:
    print("Missing columns for scatter:", xcol, ycol)

### Step 6: Molecular Visualization Engine

This step moves beyond numbers and graphs to reveal the actual chemical structures of the molecules in your library, allowing for a direct "visual audit" of the chemical scaffolds. This step transforms abstract **SMILES** strings into 2D structural images. By connecting to the **PubChem PUG-REST API**, we can fetch high-quality depictions of our compounds without needing to install heavy local chemoinformatics libraries like RDKit.

* The script selects a random sample of $N=24$ molecules to ensure a representative view of the library without overwhelming the API or the notebook's memory.
* The `smiles_to_cid` function sends the SMILES string to PubChem. It uses `urllib.parse.quote` to ensure special characters (like `#` for triple bonds or / for stereochemistry) are correctly interpreted by the web server.
* Once the PubChem Identifier (CID) is found, `cid_to_png` fetches a 250px PNG image of the 2D structure.
* The code includes `time.sleep` and `try/except` logic. This is critical when working with APIs to avoid being blocked ("rate-limited") and to handle cases where a specific SMILES might not be indexed in PubChem.
* Using `matplotlib.pyplot.subplot`, the script organizes the successfully retrieved images into a clean $6 \times 4$ grid, labeling each structure with its ChEMBL ID and its `Activity_Class`.

Visualizing the molecules allows you to confirm that the SMILES represent realistic drug-like compounds. It helps identify "artifacts," such as extremely large molecules or inorganic salts that might have slipped through the filtering process.

This teaches students how to use **REST APIs** as a powerful tool in bioinformatics. It demonstrates that you don't always need to perform calculations locally if a trusted public resource like the NIH (PubChem) can provide the data.

In [ ]:
# Step 6

import pandas as pd
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from urllib.parse import quote
import time

# --- Config ---
SMILES_COL = "SMILES"
ID_COL = "molecule_chembl_id"
N = 24
PER_ROW = 6
RANDOM_SEED = 0

IMG_SIZE = 250   # px
TIMEOUT = 30
SLEEP = 0.1      # para no saturar PubChem (puedes subir a 0.2 si hay rate-limit)

assert "df" in globals(), "DataFrame 'df' no está definido. Primero carga tu CSV en df."
assert SMILES_COL in df.columns, f"Columna '{SMILES_COL}' no encontrada en df."

# --- Sample molecules ---
df_show = (
    df.dropna(subset=[SMILES_COL])
      .sample(n=min(N, len(df)), random_state=RANDOM_SEED)
      .copy()
      .reset_index(drop=True)
)

def smiles_to_cid(smiles: str, timeout=30):
    """
    Convierte SMILES -> CID (primer CID devuelto) usando PUG-REST.
    Usa URL-encoding para evitar errores con caracteres especiales.
    """
    smi = quote(smiles, safe="")  # encode TODO (incluye / y #)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smi}/cids/JSON"
    r = requests.get(url, timeout=timeout)
    if r.status_code != 200:
        return None
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return int(cids[0]) if cids else None

def cid_to_png(cid: int, size=250, timeout=30):
    """
    Baja la imagen 2D PNG desde PubChem por CID.
    """
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/PNG?image_size={size}x{size}"
    r = requests.get(url, timeout=timeout)
    if r.status_code != 200:
        return None
    return Image.open(BytesIO(r.content)).convert("RGBA")

images, labels = [], []
failed = 0

for _, row in df_show.iterrows():
    smi_raw = str(row[SMILES_COL]).strip()
    if not smi_raw:
        continue

    cid = smiles_to_cid(smi_raw, timeout=TIMEOUT)
    if cid is None:
        failed += 1
        time.sleep(SLEEP)
        continue

    img = cid_to_png(cid, size=IMG_SIZE, timeout=TIMEOUT)
    if img is None:
        failed += 1
        time.sleep(SLEEP)
        continue

    lbl = str(row.get(ID_COL, f"CID:{cid}"))
    if "Activity_Class" in df_show.columns:
        lbl = f"{lbl}\n{row.get('Activity_Class','')}"
    else:
        lbl = f"{lbl}\nCID:{cid}"

    images.append(img)
    labels.append(lbl)

    time.sleep(SLEEP)

# --- Plot grid ---
n = len(images)
print(f"Images retrieved: {n}/{len(df_show)} | Failed: {failed}")

if n == 0:
    # Debug rápido: imprime 3 SMILES para revisar que hay contenido real
    print("Ejemplos de SMILES usados (primeros 3):")
    print(df_show[SMILES_COL].head(3).tolist())
else:
    ncols = PER_ROW
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=(ncols * 3, nrows * 3))

    for i, (img, lbl) in enumerate(zip(images, labels), start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    

### Step 7: Batch Processing

This block implements a **Batch Visualization Strategy**. Instead of looking at a random sample, this code systematically processes the entire dataset in manageable pages. This is a professional-grade approach to performing a Visual Quality Audit on a complete chemical library.

In medicinal chemistry, a "representative sample" is often not enough. This code allows the researcher to perform a **Structural Audit** of every single molecule in the library to identify rare scaffolds or unusual chemical features.

This step teaches the concept of Batching, a fundamental skill in bioinformatics and large-scale data science for handling thousands of entries without crashing the environment.


#### Technical Explanation:
1. Pagination Logic: The `for start in range(0, total, PER_PAGE)` loop slices the DataFrame into segments of 24 molecules. This prevents memory overflow and keeps the notebook organized.
2. API Integration Pipeline:
    - Translation: Converts SMILES strings into PubChem CIDs (Compound Identifiers).
    - Retrieval: Fetches the official 2D PNG coordinate-based render from the NCBI servers.
3. Rate Limiting (`time.sleep`): A deliberate pause of 0.05 seconds is included to respect PubChem's server traffic policies (PUG-REST), ensuring the script isn't flagged as a malicious bot.
4. Dynamic Grid Generation: The script calculates the required number of rows (`nrows`) for each batch and creates a new Matplotlib figure for every "page" of results, including a clear title (e.g., "Molecules 1–24").


In [ ]:
# Step 7 
## It might take a while to retrieve all images, so we won't run this by default.

import pandas as pd
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from urllib.parse import quote
import time

# --- Config ---
SMILES_COL = "SMILES"
ID_COL = "molecule_chembl_id"

PER_PAGE = 24      # how many molecules per figure
PER_ROW = 6
IMG_SIZE = 250
TIMEOUT = 30
SLEEP = 0.05

assert "df" in globals(), "DataFrame 'df' not defined."
assert SMILES_COL in df.columns, f"Column '{SMILES_COL}' not found."

df_show = df.dropna(subset=[SMILES_COL]).reset_index(drop=True)

def smiles_to_cid(smiles):
    smi = quote(smiles, safe="")
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smi}/cids/JSON"
    r = requests.get(url, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return int(cids[0]) if cids else None

def cid_to_png(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/PNG?image_size={IMG_SIZE}x{IMG_SIZE}"
    r = requests.get(url, timeout=TIMEOUT)
    if r.status_code != 200:
        return None
    return Image.open(BytesIO(r.content)).convert("RGBA")

# --- Loop over dataset in batches ---
total = len(df_show)
print(f"Total molecules: {total}")

for start in range(0, total, PER_PAGE):
    end = min(start + PER_PAGE, total)
    batch = df_show.iloc[start:end]

    images = []
    labels = []

    for _, row in batch.iterrows():
        smi = str(row[SMILES_COL]).strip()
        cid = smiles_to_cid(smi)
        if cid is None:
            continue

        img = cid_to_png(cid)
        if img is None:
            continue

        lbl = str(row.get(ID_COL, f"CID:{cid}"))
        if "Activity_Class" in row:
            lbl = f"{lbl}\n{row.get('Activity_Class','')}"

        images.append(img)
        labels.append(lbl)

        time.sleep(SLEEP)

    # --- Plot this batch ---
    n = len(images)
    ncols = PER_ROW
    nrows = int(np.ceil(n / ncols))

    plt.figure(figsize=(ncols * 3, nrows * 3))
    for i, (img, lbl) in enumerate(zip(images, labels), start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Molecules {start+1}–{end}", fontsize=14)
    plt.tight_layout()
    plt.show()



---

### EXERCISES

Test your skills with these three challenges. Use the code cells below to write your solutions.

---

**Exercise 1: Class Balance**
In the code where we plotted the Activity_Class bar chart, suppose your results showed 350 Inactive and only 5 Active molecules.

* **Question**: Is this dataset "balanced" or "imbalanced"?

* **Scientific Insight**: How might this extreme ratio affect a Machine Learning model's ability to "learn" what an active molecule looks like?

---

**Exercise 2: The Logarithmic Logic**
Look at the histogram of log10(IC50).

* **Question 1**: If a molecule has a `standard_value` of 100 nM, what is its value in `log10`? If another has 10,000 nM, what is its `log10` value?
* **Question 2**: Why do we use `np.log10` instead of plotting the raw nanomolar (nM) values directly on the x-axis?

---

**Exercise 3: Chemical Space Boundaries**
- Re-examine the AlogP vs. TPSA scatter plot.
- Locate the "Active" molecules. Are they concentrated in one specific area, or are they scattered randomly among the "Inactives"?

* **Question**: If all "Active" molecules have a TPSA between 40 and 80, what does this tell you about the oxygen/nitrogen content required for a molecule to bind to the PPAR$\gamma$ target?


---

**Exercise 4: Debugging the API Pipeline**
In the final batch visualization code (Step 7.3), we use a `time.sleep(0.05)`.

* **Question**: If you remove this line and the requests.get starts returning Error 429: Too Many Requests, what happened? How would you modify the SLEEP variable to be "safer" for the PubChem servers?


---

**Exercise 5: Structural Comparison (Visual Audit)**
Run the batch visualizer and look at the first 24 molecules. Find two "Active" molecules and two "Inactive" molecules.

* **Question**: Do you notice a specific "ring system" or a specific "side chain" (like a carboxylic acid) that appears more frequently in the Actives? This is the beginning of Manual Pharmacophore Identification.

---


### Outlook

You have completed **Molecular Visualization phase** whith a little bunch of the *Exploratory Data Analysis (EDA)* that will review later in Module 2. By moving beyond raw numbers and into the visual realm of chemistry, you have transformed a flat database into a multidimensional "Chemical Space."

From here, you now possess the skills to:
* Performed a rigorous quality audit, ensuring zero duplicates and confirming the integrity of the bioactivity labels.
* Analyze the distribution of $IC_{50}$ values, identifying the "potency spread" and validating the threshold for active compounds. 
* Through histograms and scatter plots, you can map the Lipinski-like properties (AlogP, TPSA, MW), discovering the "structural envelope" required for PPAR$\gamma$ binding.
* Integrate the PubChem API to render 2D structures, allowing for a manual inspection of the molecular scaffolds across the entire library.


See you in the next lesson!

---


© 2026 Flavio F. Contreras-Torres — MIT License